# Machine Learning on Embeddings for Epitope Prediction

The goal of this notebook is to put together a basic machine learning pipeline that can make epitope predictions using embeddings from AF3 and ESM

## Environment:

This notebook will run with the 'envs/env.yaml` environment (epident-experiments)

In [ ]:
###################
# --- Imports --- #
###################

import pickle
import sys
import os
from pathlib import Path

import polars as pl
import polars.selectors as cs
import pandas as pd
import torch
import numpy as np

from mdaf3.AF3OutputParser import AF3Output
from MDAnalysis.lib import distances
import networkx as nx

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler  
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

from plotnine import *
import matplotlib.pyplot as plt
theme_set(theme_classic())

ESM_ENCODING_DIR = Path("/tgen_labs/altin/esm_encodings")
INF_DIR = Path("../data/bp3c50id/inference")

NUM_ESM_EMB_VARS = 1280
NUM_AF3_EMB_VARS = 348

#################################
# --- Import Bepipred3 Data --- #
#################################
#   - job_name: unique identifier for protein, comes from hash of seq
#   - seq: amino acid sequence of protein
#   - train: boolean indicating if seq is part of train set
#   - epitope_boolmask: boolean array the same length as seq indiciating if the AA at that position is an epitope residue
#   - raw_protein_id: original ID assigned to protein in BP3C50ID set
#   - RSA: relative solvent accessiblity of the protein at each AA, calculated by FreeSASA
#   - SA: absolute solvent accessibility of the protein at each AA, calculated by FreeSASA

bp3 = pl.read_parquet("../data/bp3c50id/bp3c50id.rsa.parquet")

# train and test labels were swapped
bp3 = bp3.rename({"test" : "train"})
bp3 = bp3.with_row_index()

########################
# --- Num Residues --- #
########################

if "seq_len" not in bp3.columns:
    seq_lens = []
    for cols in bp3.iter_rows(named=True):
        seq_len = len(cols['seq'])
        seq_lens.append(seq_len)
    seq_lens = pl.Series("seq_len", seq_lens)
    bp3.insert_column(3, seq_lens)

#####################
# --- AF3 PTM --- #
#####################

if "ptms" not in bp3.columns:
    ptms = []
    for cols in bp3.iter_rows(named=True):
        job_name = cols['job_name']
        af3_output = AF3Output(INF_DIR / job_name)
        ptm = af3_output.get_summary_metrics()['ptm']
        ptms.append(ptm)
    ptms = pl.Series("ptm", ptms)
    bp3.insert_column(1, ptms)

#####################
# --- AF3 pLDDT --- #
#####################

if "ptms" not in bp3.columns:
    pLDDTs = []
    for cols in bp3.iter_rows(named=True):
        job_name = cols['job_name']
        af3_output = AF3Output(INF_DIR / job_name)
        u = af3_output.get_mda_universe()
        ca_atoms = u.select_atoms("protein and name CA")
        cur_protein_pLDDTs = []
        for residue in ca_atoms:
            pLDDT = residue.tempfactor
            cur_protein_pLDDTs.append(pLDDT)
        pLDDTs.append(cur_protein_pLDDTs)
    pLDDTs = pl.Series("pLDDT", pLDDTs)
    bp3.insert_column(1, pLDDTs)

##########################
# --- ESM Embeddings --- #
##########################

if "esm_emb" not in bp3.columns:
    esm_embeddings = []
    for cols in bp3.iter_rows(named=True):
        job_name = cols['job_name']
        # remove last column of embedding (sequence lengths)
        embedding = torch.load(ESM_ENCODING_DIR / (job_name + ".pt"))[:,:-1]
        esm_embeddings.append(embedding)
    esm_emb = pl.Series("esm_emb", esm_embeddings)
    bp3.insert_column(1, esm_emb)

##########################
# --- AF3 Embeddings --- #
##########################

if "af_emb" not in bp3.columns:
    af_embeddings = []
    for cols in bp3.iter_rows(named=True):
        job_name = cols['job_name']
        af3_output = AF3Output(INF_DIR / job_name)
        af3_single_embed = af3_output.get_single_embeddings()
        # remove alphafold embedding padding 
        num_tokens = cols['seq_len']
        if af3_single_embed.shape[0] != num_tokens:
            diff = int(af3_single_embed.shape[0] - num_tokens)
            af_emb = af3_single_embed[:-diff,:]
        af3_tensor = torch.from_numpy(af_emb)
        af_embeddings.append(af3_tensor)
    af_emb = pl.Series("af_emb", af_embeddings)
    bp3.insert_column(1, af_emb)
    # TEMP: for some reason, row 192 has an esm embedding smaller than af3 embedding... 
    bp3 = bp3.filter(pl.col("index") != 192).drop("index")
    bp3 = bp3.with_row_index()

#########################################
# --- AF3 Protein Structure Network --- #
#########################################
# For each AF3 structure, we generalize a graph whose nodes are residues
# and whose edges are the distance between residues. Edges are only drawn when
# the probability of contact exceeds the threshold specified (0.5) AND the
# predicted alignment error falls below the cutoff (5).

#cutoff_dist = 10    # Angstroms
cutoff_prob = 0.825   # Probability of Contact
cutoff_pae = 10      # Predicted Alignment Error

if "af_graph" not in bp3.columns:
    af_graphs = []
    for cols in bp3.iter_rows(named=True):
        job_name = cols['job_name']
        af3_output = AF3Output(INF_DIR / job_name)
        u = af3_output.get_mda_universe()

        ca_atoms = u.select_atoms('protein and name CA')
        ca_positions = ca_atoms.positions

        dist_array_flat = distances.self_distance_array(ca_positions)
        n_residues = len(ca_atoms)
        distance_matrix = np.zeros((n_residues, n_residues))
        triu_indices = np.triu_indices(n_residues, k=1)
        distance_matrix[triu_indices] = dist_array_flat
        distance_matrix.T[triu_indices] = dist_array_flat

        contact_probability_matrix = af3_output.get_contact_prob_ndarr()
        pae_mtx = af3_output.get_pae_ndarr()

        cont_adj_mtx = (contact_probability_matrix >= cutoff_prob).astype(int)
        pae_adj_mtx = (pae_mtx <= cutoff_pae).astype(int)
        adj_mtx = distance_matrix * pae_adj_mtx * cont_adj_mtx
        np.fill_diagonal(adj_mtx, 0)

        resids = ca_atoms.resids
        G = nx.Graph()
        for i, resid in enumerate(resids):
            G.add_node(resid)
            
        af_graphs.append(G)

        # Iterate over the upper triangle of the adjacency matrix to find contacts (edges)
        for i in range(n_residues):
            for j in range(i + 1, n_residues):
                if adj_mtx[i, j] > 1:
                    distance = distance_matrix[i, j]
                    G.add_edge(resids[i], resids[j], weight=distance)
    af_graphs = pl.Series("af_graph", af_graphs)
    bp3.insert_column(1, af_graphs)

    
###############################################
# --- Analysis of AF3 Network's Structure --- #
###############################################

closeness_centrality = []
betweenness_centrality = []
load_centrality = []
eigenvector_centrality = []
degree_centrality = []
clustering = []
coreness = []
triangles = []
density = []
lapl_n1 = []
lapl_f = []
induced_subgraphs = []
SUBGRAPH_DISTANCE_CUTOFF = 15
for cols in bp3.iter_rows(named=True):
    af_graph = cols['af_graph']
    closeness_centrality.append(pl.Series(list(nx.closeness_centrality(af_graph).values())))
    betweenness_centrality.append(pl.Series('betweenness_centrality', list(nx.betweenness_centrality(af_graph).values())))
    load_centrality.append(pl.Series('load_centrality', list(nx.load_centrality(af_graph).values())))
    eigenvector_centrality.append(pl.Series('eigenvector_centrality', list(nx.eigenvector_centrality(af_graph, max_iter=10000).values())))
    degree_centrality.append(pl.Series('degree_centrality', list(nx.degree_centrality(af_graph).values())))
    clustering.append(pl.Series('clustering', list(nx.clustering(af_graph).values()), dtype=pl.Float64))
    coreness.append(pl.Series('coreness', list(nx.core_number(af_graph).values()), dtype=pl.Float64))
    triangles.append(pl.Series('triangles', list(nx.triangles(af_graph).values()), dtype=pl.Float64))
    density.append(nx.density(af_graph))

    lapl_mtx = nx.laplacian_matrix(af_graph).toarray()
    eigvals, _ = np.linalg.eig(lapl_mtx)
    # remove near 0 eigvals to pull fielder val
    eigvals = np.real(eigvals[np.abs(eigvals) >= 1e-3])
    lapl_f.append(np.min(eigvals))
    lapl_n1.append(np.max(eigvals))
closeness_centrality = pl.Series('closeness_centrality', closeness_centrality)
betweenness_centrality = pl.Series('betweenness_centrality', betweenness_centrality)
load_centrality = pl.Series('load_centrality', load_centrality)
eigenvector_centrality = pl.Series('eigenvector_centrality', eigenvector_centrality)
degree_centrality = pl.Series('degree_centrality', degree_centrality)
clustering = pl.Series('clustering', clustering)
coreness = pl.Series('coreness', coreness)
triangles = pl.Series('triangles', triangles)
density = pl.Series('density', density)
lapl_n1 = pl.Series("lapl_n1", lapl_n1)
lapl_f = pl.Series("lapl_f", lapl_f)

graph_features = pl.DataFrame([
    closeness_centrality, betweenness_centrality, load_centrality, eigenvector_centrality, degree_centrality, 
    clustering, coreness, triangles, density, lapl_f, lapl_n1]).with_row_index()

if 'clustering' not in bp3.columns:
    bp3 = bp3.join(graph_features, on='index', how='full').drop(['index_right'])

##########################################
# --- Transform to Per-Residue Basis --- #
##########################################

af_emb = []
esm_emb = []
af_graph = []
epitope = []
rsa = []
sa = []
closeness_centrality = []
betweenness_centrality = []
load_centrality = []
eigenvector_centrality = []
degree_centrality = []
clustering = []
coreness = []
triangles = []
pLDDT = []
ptm = []
job_name = []
seq = []
seq_len = []
train = []
raw_protein_id = []
density = []
lapl_f = []
lapl_n1 = []
bp3_res = bp3.drop("index")
for cols in bp3.iter_rows(named=True):
    # Residue Features
    af_emb.extend(cols['af_emb'])
    esm_emb.extend(cols['esm_emb'])
    epitope.extend(cols['epitope_boolmask'])
    rsa.extend(cols['RSA'])
    sa.extend(cols['SA'])
    closeness_centrality.extend(cols['closeness_centrality'])
    betweenness_centrality.extend(cols['betweenness_centrality'])
    load_centrality.extend(cols['load_centrality'])
    eigenvector_centrality.extend(cols['eigenvector_centrality'])
    degree_centrality.extend(cols['degree_centrality'])
    clustering.extend(cols['clustering'])
    coreness.extend(cols['coreness'])
    triangles.extend(cols['triangles'])
    pLDDT.extend(cols['pLDDT'])

    # Global Features
    for repeats in range(cols['seq_len']):
        af_graph.append(cols['af_graph'])
        ptm.append(cols['ptm'])
        job_name.append(cols['job_name'])
        seq.append(cols['seq'])
        seq_len.append(cols['seq_len'])
        train.append(cols['train'])
        raw_protein_id.append(cols['raw_protein_id'])
        density.append(cols['density'])
        lapl_f.append(cols['lapl_f'])
        lapl_n1.append(cols['lapl_n1'])

af_emb = pl.Series('af_emb', af_emb)
esm_emb = pl.Series('esm_emb', esm_emb)
af_graph = pl.Series('af_graph', af_graph)
epitope = pl.Series('epitope', epitope)
rsa = pl.Series('rsa', rsa)
sa = pl.Series('sa', sa)
closeness_centrality = pl.Series('closeness_centrality', closeness_centrality)
betweenness_centrality = pl.Series('betweenness_centrality', betweenness_centrality)
load_centrality = pl.Series('load_centrality', load_centrality)
eigenvector_centrality = pl.Series('eigenvector_centrality', eigenvector_centrality)
degree_centrality = pl.Series('degree_centrality', degree_centrality)
clustering = pl.Series('clustering', clustering)
coreness = pl.Series('coreness', coreness)
triangles = pl.Series('triangles', triangles)
pLDDT = pl.Series('pLDDT', pLDDT)
ptm = pl.Series('ptm', ptm)
job_name = pl.Series('job_name', job_name)
seq = pl.Series('seq', seq)
seq_len = pl.Series('seq_len', seq_len)
train = pl.Series('train', train)
raw_protein_id = pl.Series('raw_protein_id', raw_protein_id)
density = pl.Series('density', density)
lapl_f = pl.Series('lapl_f', lapl_f)
lapl_n1 = pl.Series('lapl_n1', lapl_n1)

bp3_res = pl.DataFrame([
    job_name, raw_protein_id, seq, seq_len, esm_emb, af_emb, af_graph,
    closeness_centrality, betweenness_centrality, load_centrality, eigenvector_centrality,
    degree_centrality, clustering, coreness, triangles, density, lapl_f, lapl_n1, 
    ptm, pLDDT, rsa, sa, epitope, train
    ]).with_row_index()

#####################################
# --- Explode Embedding Columns --- #
#####################################

bp3_esm_res = bp3_res.select(
    pl.col('index'),
    pl.col("esm_emb").map_batches(
        lambda s: pl.Series(
            np.stack([t.cpu().numpy() for t in s.to_list()]),
            dtype=pl.List(pl.Float64)
        ),
        return_dtype=pl.List(pl.Float64)
    )
).with_columns(
    pl.col("esm_emb").list.to_struct(
        fields=[f"esm_{i}" for i in range(NUM_ESM_EMB_VARS)]
    )
).unnest("esm_emb")

bp3_af3_res = bp3_res.select(
    pl.col("index"),
    pl.col("af_emb").map_batches(
        lambda s: pl.Series(
            np.stack([t.cpu().numpy() for t in s.to_list()]),
            dtype=pl.List(pl.Float64)
        ),
        return_dtype=pl.List(pl.Float64)
    )
).with_columns(
    pl.col("af_emb").list.to_struct(
        fields=[f"af3_{i}" for i in range(NUM_AF3_EMB_VARS)]
    )
).unnest("af_emb")

bp3_no_emb_res = bp3_res.drop(['af_emb', 'esm_emb'])
bp3_res = bp3_no_emb_res.join(bp3_esm_res, on='index', how='full').drop(['index_right'])
bp3_res = bp3_res.join(bp3_af3_res, on='index', how='full').drop(['index_right'])
bp3_df = bp3_res.drop('af_graph')


/home/jsesate/miniconda3/envs/epident-experiments/lib/python3.13/site-packages/MDAnalysis/coordinates/MMCIF.py:139: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.


In [ ]:
##########################
# --- Save DataFrame --- #
##########################

#with open('bp3_pae10.pkl', 'wb') as f:
#        pickle.dump(bp3_df, f)

with open("bp3_pae10.pkl", 'rb') as f:
        bp3_all_df = pickle.load(f)
bp3_train_df = bp3_all_df.filter(pl.col("train") == True)
print(f"Num Epitope Residues: {len(bp3_train_df.filter(pl.col("epitope") == True))}")
print(f"Num Non-Epitope Residues: {len(bp3_train_df.filter(pl.col("epitope") == False))}")

print(f"Column Names: {bp3_all_df.columns}")
bp3_test_df = bp3_all_df.filter(pl.col("train") == False)

Num Epitope Residues: 10811
Num Non-Epitope Residues: 70702
Column Names: ['index', 'job_name', 'raw_protein_id', 'seq', 'seq_len', 'closeness_centrality', 'betweenness_centrality', 'load_centrality', 'eigenvector_centrality', 'degree_centrality', 'clustering', 'coreness', 'triangles', 'density', 'lapl_f', 'lapl_n1', 'ptm', 'pLDDT', 'rsa', 'sa', 'epitope', 'train', 'esm_0', 'esm_1', 'esm_2', 'esm_3', 'esm_4', 'esm_5', 'esm_6', 'esm_7', 'esm_8', 'esm_9', 'esm_10', 'esm_11', 'esm_12', 'esm_13', 'esm_14', 'esm_15', 'esm_16', 'esm_17', 'esm_18', 'esm_19', 'esm_20', 'esm_21', 'esm_22', 'esm_23', 'esm_24', 'esm_25', 'esm_26', 'esm_27', 'esm_28', 'esm_29', 'esm_30', 'esm_31', 'esm_32', 'esm_33', 'esm_34', 'esm_35', 'esm_36', 'esm_37', 'esm_38', 'esm_39', 'esm_40', 'esm_41', 'esm_42', 'esm_43', 'esm_44', 'esm_45', 'esm_46', 'esm_47', 'esm_48', 'esm_49', 'esm_50', 'esm_51', 'esm_52', 'esm_53', 'esm_54', 'esm_55', 'esm_56', 'esm_57', 'esm_58', 'esm_59', 'esm_60', 'esm_61', 'esm_62', 'esm_63', 'e

In [ ]:
#######################################################################
# --- Visualize Variable Distributions for Epitope vs. NonEpitope --- #
#######################################################################

bp3_plot = bp3_train_df.to_pandas()
plot_var = "pLDDT"
(           #
ggplot(aes(x = bp3_plot["epitope"], y = bp3_plot[plot_var]))
+ geom_boxplot()                                        
+ labs(
    x = "Epitope Status",
    y = f"{plot_var}"
)
#+ geom_jitter()
)

In [ ]:
#############################
# --- Feature Selection --- #
#############################

agg_features = []

for emb in range(NUM_ESM_EMB_VARS):
    esm_vars = f"esm_{emb}"
    agg_features.append(esm_vars)

for emb in range(NUM_AF3_EMB_VARS):
    af3_vars = f"af3_{emb}"
    agg_features.append(af3_vars)

#agg_features.extend([
#    'closeness_centrality', 'betweenness_centrality', 'load_centrality', 
#    'eigenvector_centrality', 'degree_centrality', 'clustering', 
#    'coreness', 'triangles', 'density', 'lapl_n1', 'lapl_f'])
agg_features.append("seq_len") 
#agg_features.append('pLDDT')
agg_features.append("ptm") 
agg_features.append("rsa")
agg_features.append("sa")

print(f"Num Features: {len(agg_features)}")

Num Features: 1632


In [187]:
###################################
# --- 5-Fold Cross Validation --- #
###################################

train_df = bp3_train_df.to_pandas()
X_df = train_df[agg_features]
y_df = train_df["epitope"]

X = X_df.values
y = y_df.values

n_splits = 5
cv = KFold(
    n_splits=n_splits, 
    shuffle=False,
    #random_state=11
    )

train_auc_scores = []
test_auc_scores = []
components = []

print("--- Cross-Validation Fold Details ---")
for fold, (train_index, test_index) in enumerate(cv.split(X, y)):

    # --- Cross Validation ---
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # --- Scale Features (Required for PCA) ---
    scaler = StandardScaler()  
    scaler.fit(X_train)  
    X_train = scaler.transform(X_train)  
    X_test = scaler.transform(X_test)  

    # --- Calibrate PCA ---
    pca = PCA(n_components=None, random_state=11) # n_components=None keeps all 1280 components
    pca.fit(X_train)

    # Calculate the cumulative explained variance
    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
    variance_threshold =  0.95
    optimal_k = np.where(cumulative_variance >= variance_threshold)[0][0] + 1
    components.append(optimal_k)
    pca_final = PCA(n_components=optimal_k, random_state=11)

    # --- Enable PCA ---
    #pca_final = PCA(n_components=1111, random_state=11)
    X_train = pca_final.fit_transform(X_train)
    X_test = pca_final.transform(X_test)

    # Lasso Regularization (best so far)
    clf = LogisticRegression(solver="saga", class_weight="balanced", penalty='l1', C=0.0025, max_iter=500, n_jobs=-1, random_state=11)

    # Ridge Regularization
    #clf = LogisticRegression(solver="saga", class_weight="balanced", penalty='l2', C=0.000025, max_iter=500, n_jobs=-1, random_state=11)

    clf.fit(X_train, y_train)

    # --- Training AUC Calculation ---
    y_train_proba = clf.predict_proba(X_train)[:, 1]
    train_auc = roc_auc_score(y_train, y_train_proba)
    train_auc_scores.append(train_auc)

    # --- Test AUC Calculation ---
    y_test_proba = clf.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, y_test_proba)
    test_auc_scores.append(test_auc)

    print(f"Fold {fold+1}: Train AUC = {train_auc:.4f}, Test AUC = {test_auc:.4f}")

# Mean ROC data
mean_auc_test = np.mean(test_auc_scores)
std_auc_test = np.std(test_auc_scores)

# Mean PCA Components
mean_components = np.mean(components)
std_components = np.std(components)

# --- Overfitting Check Section ---
print("\n--- Overfitting Check ---")
mean_train_auc = np.mean(train_auc_scores)
std_train_auc = np.std(train_auc_scores)

print(f"PCA Variance Kept: {variance_threshold*100}% with {mean_components:.4f} (+/- {std_components:.4f}) components")

print(f"Average Training AUC across folds: {mean_train_auc:.4f} (+/- {std_train_auc:.4f})")
print(
    f"Average Test (Validation) AUC across folds: {mean_auc_test:.4f} (+/- {std_auc_test:.4f})"
)

--- Cross-Validation Fold Details ---
Fold 1: Train AUC = 0.7720, Test AUC = 0.7450
Fold 2: Train AUC = 0.7781, Test AUC = 0.7380
Fold 3: Train AUC = 0.7792, Test AUC = 0.7384
Fold 4: Train AUC = 0.7748, Test AUC = 0.7583
Fold 5: Train AUC = 0.7717, Test AUC = 0.7499

--- Overfitting Check ---
PCA Variance Kept: 95.0% with 1103.6000 (+/- 3.3823) components
Average Training AUC across folds: 0.7752 (+/- 0.0031)
Average Test (Validation) AUC across folds: 0.7459 (+/- 0.0076)


In [97]:
################################
# --- BP3 Final Evaluation --- #
################################

train_df = bp3_train_df.to_pandas()
X_train = train_df[agg_features]
y_train = train_df["epitope"]

test_df = bp3_test_df.to_pandas()
X_test = test_df[agg_features]
y_test = test_df["epitope"]

# --- Scale Features (Required for PCA) ---
scaler = StandardScaler()  
scaler.fit(X_train)  
X_train = scaler.transform(X_train)  
X_test = scaler.transform(X_test)  

# --- PCA ---
pca = PCA(n_components=None, random_state=11) # n_components=None keeps all 1280 components
pca.fit(X_train)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
variance_threshold =  0.95
optimal_k = np.where(cumulative_variance >= variance_threshold)[0][0] + 1
components.append(optimal_k)
pca_final = PCA(n_components=optimal_k, random_state=11)
X_train = pca_final.fit_transform(X_train)
X_test = pca_final.transform(X_test)

# --- Fit Model ---
clf = LogisticRegression(solver="saga", class_weight="balanced", penalty='l1', C=0.0025, max_iter=500, n_jobs=-1, random_state=11)
clf.fit(X_train, y_train)

# --- Training AUC Calculation ---
y_train_proba = clf.predict_proba(X_train)[:, 1]
train_auc = roc_auc_score(y_train, y_train_proba)
train_auc_scores.append(train_auc)

# --- Test AUC Calculation ---
y_test_proba = clf.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_test_proba)
test_auc_scores.append(test_auc)

print(f"PCA Variance Kept: {variance_threshold*100}% with {optimal_k} components")
print(f"Train AUC = {train_auc:.4f}, Test AUC = {test_auc:.4f}")

PCA Variance Kept: 95.0% with 1024 components
Train AUC = 0.7421, Test AUC = 0.7703
